# DataMate · Baseline text-to-SQL (Zalo AI Challenge 2026, Track 2)

**Bài toán:** trả lời câu hỏi tiếng Việt về dữ liệu kinh doanh của một cửa hàng (các bảng csv / xlsx / sqlite và một README). Đáp án có kiểu
cố định (số, phần trăm, chuỗi, danh sách, ngày, bảng…), hoặc bỏ trống khi không chắc.

**Ý tưởng của baseline:**
- Mô hình ngôn ngữ (Qwen3.5-9B + LoRA) **viết SQL**, còn DuckDB **thực hiện phép tính** trên dữ liệu thật.
- Mỗi câu hỏi lấy nhiều mẫu SQL, chạy từng mẫu rồi **bỏ phiếu** theo đáp án. Khi các mẫu không đồng thuận, hệ thống **bỏ trống**
  thay vì đoán, vì câu trả lời sai bị trừ điểm.
- LoRA được fine-tune bằng **SFT LoRA BF16**. Dữ liệu mặc định là bộ SFT tổng hợp trong `data/data/` (shop và câu hỏi được sinh ra, đã chia
  train / dev theo shop). Phần 04 là cách thứ hai: dựng dữ liệu từ chính các lời giải đúng mà mô hình tìm ra trên tập train (rejection sampling),
  giữ 6 shop holdout ngoài dữ liệu train để đo sạch.

| Phần | Nội dung | Phần cứng |
|---|---|---|
| 01 · EDA | dữ liệu gồm những gì: shop, tệp, kiểu đáp án, dạng câu hỏi, rulebook | CPU |
| 02 · Suy luận | chạy pipeline và ghi file nộp `submission.jsonl` | GPU bf16 từ 24 GB |
| 03 · Đánh giá | điểm PA, nhóm lỗi, ngưỡng bỏ phiếu, so hai adapter trên cùng câu hỏi | CPU |
| 04 · Dữ liệu SFT | dựng dữ liệu SFT từ các lời giải đúng, chia train / dev theo shop (không bắt buộc khi dùng bộ SFT `data/data`) | CPU, GPU cho bước ghi chú |
| 05 · Fine-tune | SFT LoRA BF16 trên Qwen3.5-9B | GPU bf16 từ 40 GB |

Chạy từ trên xuống. Mỗi phần có một ô tham số ở đầu phần. Phần 01-04 cần dữ liệu BTC (`data/train`, `data/public_test` hoặc `DATAMATE_DATA`),
phần 05 chỉ cần bộ SFT `data/data` (chỉ train thì mở `05_train_lora` riêng). Khi không có GPU đủ lớn (ví dụ T4 trên Kaggle), các bước cần mô hình được bỏ qua
hoặc dùng kết quả đã chạy sẵn nếu checkout có (`runs/saved`).

In [ ]:
# Setup: tìm checkout của repo này, nạp datamate.notebook, phát hiện dữ liệu, GPU, thư mục model và runs.
#   local   mở notebook trong notebooks/ của checkout (hoặc đặt biến môi trường DATAMATE_REPO), không cần cài thêm.
#   Kaggle  đính kèm bản sao checkout làm dataset (Add data) cùng dữ liệu BTC (dataset riêng tư), bật Internet để cài gói.
#   Colab   chép checkout vào MyDrive/datamate/code, dữ liệu BTC vào MyDrive/datamate/data.
# Kaggle / Colab: code được chép sang ổ ghi được (input của Kaggle chỉ đọc, Drive chậm); kết quả ghi vào env.runs, được giữ sau phiên.
import os, shutil, sys
from pathlib import Path

MARKER = "datamate/notebook.py"


def find_checkout() -> Path:
    if os.environ.get("DATAMATE_REPO"):
        return Path(os.environ["DATAMATE_REPO"]).resolve()
    for c in (Path.cwd(), Path.cwd().parent):
        if (c / MARKER).exists():
            return c.resolve()
    if Path("/kaggle/input").exists():
        found, work = sorted(p.parents[1] for p in Path("/kaggle/input").glob(f"**/{MARKER}")), Path("/tmp/datamate")
    elif "COLAB_RELEASE_TAG" in os.environ:
        from google.colab import drive
        drive.mount("/content/drive")
        found, work = [p for p in [Path("/content/drive/MyDrive/datamate/code")] if (p / MARKER).exists()], Path("/content/datamate")
    else:
        found, work = [], None
    if not found:
        raise FileNotFoundError("không tìm thấy checkout datamate: mở notebook trong notebooks/ của repo, hoặc đặt DATAMATE_REPO")
    if not (work / MARKER).exists():
        shutil.copytree(found[0], work, ignore=shutil.ignore_patterns(".git", "runs", "models", ".venv*", "__pycache__"))
    return work


root = find_checkout()
os.chdir(root)
sys.path.insert(0, str(root))
from datamate.notebook import install, setup
install("base")   # mỗi phần bên dưới cài thêm thứ nó cần (chỉ Kaggle / Colab)
env = setup()

In [ ]:
import json
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

read = lambda p: [json.loads(l) for l in open(p, encoding="utf-8-sig") if l.strip()]
pd.set_option("display.max_colwidth", 80)

## 01 · Khám phá dữ liệu (EDA)

Trước khi xây pipeline, ta xem dữ liệu trông như thế nào:
- mỗi cửa hàng (shop) có bao nhiêu câu hỏi và những tệp dữ liệu gì
- đáp án có những kiểu nào, câu hỏi thường hỏi gì
- rulebook quy định cách tính (doanh thu, hàng bán chạy, công nợ…) ra sao
- một shop sau khi nạp vào DuckDB trông thế nào: đây chính là phần schema mà mô hình nhìn thấy

In [ ]:
env.split("train"), env.split("public_test")   # dữ liệu BTC (data/train, data/public_test): báo rõ chỗ đặt khi thiếu
train_q, public_q = read(env.data / "train/questions.jsonl"), read(env.data / "public_test/questions.jsonl")
train_a = {a["id"]: a for a in read(env.data / "train/answers.jsonl")}
splits = {"train": train_q, "public": public_q}
def per_shop(qs):
    c = list(Counter(q["business_id"] for q in qs).values())
    return f"{min(c)} / {pd.Series(c).median():.0f} / {max(c)}"
pd.DataFrame({name: {"questions": len(qs), "shops": len({q["business_id"] for q in qs}), "questions per shop (min / median / max)": per_shop(qs),
                     "as_of_date range": f'{min(q["as_of_date"] for q in qs)} .. {max(q["as_of_date"] for q in qs)}'} for name, qs in splits.items()})

### Tệp dữ liệu của shop

Mỗi shop là một thư mục gồm README.md và các bảng (csv / xlsx / sqlite). Số bảng và dung lượng khác nhau rất nhiều.

In [ ]:
rows = []
for split, folder in (("train", "train"), ("public", "public_test")):
    for shop in sorted((env.data / folder / "data").iterdir()):
        files = [f for f in shop.rglob("*") if f.is_file() and f.name != "README.md"]
        rows.append({"split": split, "shop": shop.name, "files": len(files), "MB": round(sum(f.stat().st_size for f in files) / 2**20, 1),
                     **Counter(f.suffix.lower().lstrip(".") for f in files)})
files = pd.DataFrame(rows).fillna(0)
display(files.groupby("split")[[c for c in files.columns if c not in ("split", "shop")]].sum())
files.groupby("split")[["files", "MB"]].describe().round(1)

### Loại đáp án (train)

Phân bố type của đáp án chuẩn. `alias` là các cách viết khác được chấp nhận (mã hàng, cách viết khác trong dữ liệu).

In [ ]:
types = Counter(a["type"] for a in train_a.values())
pd.Series(types).sort_values().plot.barh(figsize=(6, 3), title="train answer types")
print({t: f"{n / len(train_a):.1%}" for t, n in types.most_common()})
print("answers with aliases:", sum(bool(a.get("alias")) for a in train_a.values()))

Mỗi kiểu đáp án một ví dụ:

In [ ]:
example = {}
for q in train_q:
    a = train_a[q["id"]]
    example.setdefault(a["type"], (q["business_id"], q["question"], a.get("answer")))
pd.DataFrame([{"type": t, "shop": shop, "question": text[:100], "answer": json.dumps(ans, ensure_ascii=False)[:70]}
              for t, (shop, text, ans) in example.items()])

### Dạng câu hỏi

Tỷ lệ câu hỏi chứa một số dạng khó, ví dụ khung thời gian ("tuần rồi", "quý 2"), công nợ, số tiền viết tắt ("3tr7") hay từ địa phương.
Pipeline tính sẵn gợi ý cho các dạng này (`datamate/hints.py`) để mô hình không phải tự suy đoán.

In [ ]:
import re
from datamate import hints as H
from datamate.ingest import norm
from datamate.prompts import SLANG, money_gloss
shapes = {
    "period named": lambda q: bool(H.period_hints(q["question"], q["as_of_date"])),
    "debt": lambda q: bool(H.DEBT_WORDS.search(norm(q["question"]))),
    "days by daily total": lambda q: bool(H.COUNT_DAYS.search(norm(q["question"])) and H.THRESHOLD.search(norm(q["question"]) + q["question"])),
    "profit": lambda q: bool(H.PROFIT.search(norm(q["question"]))),
    "delivery trips": lambda q: bool(H.TRIPS.search(norm(q["question"]))),
    "lots / expiry": lambda q: bool(H.LOT.search(norm(q["question"]))),
    "money shorthand": lambda q: bool(money_gloss(q["question"])),
    "chat words": lambda q: bool(set(re.findall(r"\w+", q["question"].lower())) & set(SLANG)),
}
share = pd.DataFrame({name: {k: sum(f(q) for q in qs) / len(qs) for k, f in shapes.items()} for name, qs in splits.items()})
share.plot.barh(figsize=(7, 4), title="share of questions")
share.style.format("{:.1%}")

In [ ]:
pd.DataFrame({name: pd.Series([len(q["question"]) for q in qs]).describe().round(0) for name, qs in splits.items()}).T

### Rulebook

Train và public có rulebook riêng. Public thêm một số luật (khách trung thành, giờ cao điểm, nhà cung cấp chính…).

In [ ]:
rb = {s: (env.data / f / "rulebook.md").read_text(encoding="utf-8") for s, f in (("train", "train"), ("public", "public_test"))}
rules = {s: {l.split("|")[1].strip() for l in t.splitlines() if l.startswith("| ") and not l.startswith("| Luật")} for s, t in rb.items()}
print(len(rules["train"]), "rules in train,", len(rules["public"]), "in public")
print("only in public:", sorted(rules["public"] - rules["train"]))
print("only in train:", sorted(rules["train"] - rules["public"]))

### Một shop sau khi nạp

`load_business` đọc mọi bảng vào DuckDB, `schema_text` là phần SCHEMA mà model nhìn thấy.

In [ ]:
from datamate.ingest import load_business, schema_text
db, meta = load_business(env.data / "train", "biz_101", canon=True)
display(pd.DataFrame([{"table": t["name"], "rows": t["rows"], "columns": len(t["columns"])} for t in meta["tables"]]))
print(schema_text(meta)[:3000])

## 02 · Suy luận (inference)

Pipeline xử lý từng shop qua năm bước:
1. **Nạp dữ liệu** vào DuckDB và dựng sẵn view chuẩn (`v_sales`, `v_orders`) cùng macro công nợ, để câu SQL ngắn và ít sai.
2. **Tạo prompt** gồm rulebook, schema, khung thời gian tính sẵn, vài ví dụ tương tự và câu hỏi.
3. **Lấy mẫu:** mô hình viết 8 câu SQL. Câu khó (các mẫu không đồng thuận) được thêm 4 mẫu ở chế độ thinking.
4. **Thực thi và sửa lỗi:** DuckDB chạy từng câu SQL. Câu bị lỗi được gửi lại cho mô hình kèm thông báo lỗi để sửa.
5. **Bỏ phiếu:** gom các đáp án giống nhau. Hệ thống chỉ trả lời khi nhóm lớn nhất đủ đồng thuận, ngược lại thì bỏ trống.

Kết quả là file `submission.jsonl` đúng định dạng nộp bài. `LIMIT = 10` chạy thử 10 câu, còn cả 1000 câu public mất khoảng 3,5 giờ
trên một GPU 24 GB.

| Mục đích | `SPLIT` | `IDS` | `REUSE_LAYERS` | `LORA` | `SAVE_PROMPTS` |
|---|---|---|---|---|---|
| Bản nộp public | `"public_test"` | `None` | `True` | adapter đã chọn | tùy |
| Lượt có nhãn để dựng dữ liệu SFT (phần 04) | `"train"` | `"splits/rft_train24.txt"` | `False` | `"sql_rft_v3"` | `True` |
| Đo adapter mới trên holdout (phần 05 → 03) | `"train"` | `"splits/holdout600.txt"` | `False` | `"runs/lora/qwen35_sft_a6000/adapter"` | tùy |
| Mốc so sánh trên holdout | `"train"` | `"splits/holdout600.txt"` | `False` | `"sql_rft_v3"` | tùy |

In [ ]:
install("llm")

In [ ]:
SPLIT = "public_test"        # public_test | train | private_test (thư mục trong data/)
LIMIT = 10                   # N câu đầu (None = tất cả)
IDS = None                   # tệp id câu hỏi, ví dụ "splits/holdout600.txt" hoặc "splits/rft_train24.txt"
REUSE_LAYERS = True          # chỉ public: dùng lại view và lớp công nợ của lượt 0.683 (configs/public_v9). False = soạn mới (như private)
LORA = "sql_rft_v3"          # adapter trong adapters/, hoặc adapter từ 05_train_lora, ví dụ "runs/lora/qwen35_sft_a6000/adapter"
SAVE_PROMPTS = True          # lưu prompt trong trace (cần để dựng dữ liệu SFT từ lượt chạy train)
OUT_NAME = None              # None = nb_<split>_<tên adapter>

In [ ]:
env.split(SPLIT)                                                        # dữ liệu BTC của split này
lora_arg = env.path(LORA) if env.path(LORA).exists() else LORA            # đường dẫn (runs/..., tuyệt đối) hoặc tên trong adapters/
lora_name = Path(LORA).parent.name if Path(LORA).name == "adapter" else Path(LORA).name
out = env.runs / (OUT_NAME or f"nb_{SPLIT}_{lora_name}")
if env.can_run_llm:
    model, lora = env.base_model(), env.adapter(lora_arg)
    flags, engine = env.inference_flags()
    n = len(env.gpus)
    layers = ["--views-from", "configs/public_v9"] if REUSE_LAYERS and SPLIT == "public_test" else []
    pick = (["--limit", str(LIMIT)] if LIMIT else []) + (["--ids", env.path(IDS)] if IDS else []) + (["--save-prompts"] if SAVE_PROMPTS else [])
    make_cmd = lambda i: [env.python, "-m", "datamate.predict", "--data", env.data / SPLIT, "--output", out / f"shard{i}", "--shard", f"{i}/{n}",
                          "--model", model, "--lora", lora, "--fewshot", "assets/fewshot_bank.jsonl", *flags, "--engine", engine, *layers, *pick]
    print(f"{n} GPU process(es) -> {out}\nadapter: {lora}\nflags: {' '.join(flags)}")
else:   # T4 / P100 hoặc CPU: dùng lượt chạy y hệt đã chạy sẵn trên GPU 24 GB (public_test, LIMIT = 10), nếu checkout có
    try:
        env.use_saved("nb_" + SPLIT, out)
    except RuntimeError as e:
        print("không có GPU bf16 >= 24 GB và không có kết quả chạy sẵn:", e)

In [ ]:
if env.can_run_llm:
    env.run_shards(make_cmd, n, out)

In [ ]:
from datamate.submission import check, merge_shards

if list(out.glob("shard*/trace.jsonl")):
    merge_shards(out, env.data / SPLIT / "questions.jsonl")
    errors = check(out / "submission.jsonl", env.data / SPLIT / "questions.jsonl")
    print("format OK" if not errors else errors[:5])
else:
    print(f"chưa có lượt chạy trong {out}")

### Kết quả

Số câu đã trả lời theo shop, vài câu mẫu cùng đáp án và tỷ lệ đồng thuận (`conf`). Với `SPLIT = "train"` có thêm điểm theo đáp án chuẩn.

In [ ]:
trace = [r for p in sorted(out.glob("shard*/trace.jsonl")) for r in read(p)]
df = pd.DataFrame([{"shop": r["business_id"], "answered": r["final"]["type"] != "abstain", "conf": r["conf"]} for r in trace])
if len(df):
    display(df.groupby("shop").agg(questions=("answered", "size"), answered=("answered", "sum"), mean_conf=("conf", "mean")).round(2))
pd.DataFrame([{"question": r["question"][:90], "final": json.dumps(r["final"], ensure_ascii=False)[:80], "conf": round(r["conf"], 2)} for r in trace[:5]])

### Một câu hỏi đi qua pipeline

Câu hỏi có tỷ lệ đồng thuận thấp nhất (thường là câu khó, được thêm mẫu thinking): chế độ, câu SQL và đáp án của từng mẫu, rồi kết quả bỏ phiếu.
- Mẫu không viết ra được SQL hợp lệ (ví dụ ghi chú lặp mãi không dừng) không được tính phiếu.
- Tỷ lệ đồng thuận = số phiếu của đáp án thắng / tổng số mẫu.

In [ ]:
def last_sql(text):
    return " ".join(text.split("```sql")[-1].split("```")[0].split()) if "```sql" in text else ""


if trace:
    q = min(trace, key=lambda r: (r["conf"], -len(r["samples"])))
    print(q["question"])
    samples = pd.DataFrame([{"mẫu": k + 1,
                             "chế độ": "thinking" if "</think>" in s["text"] else "thường",
                             "đáp án": json.dumps(s["pred"]["answer"], ensure_ascii=False)[:50] if s.get("pred") else "",
                             "SQL": last_sql(s["text"])[:110] or "(lỗi: " + (s.get("error") or "").split(" | ")[0] + ")"}
                            for k, s in enumerate(q["samples"])])
    display(samples)
    votes = Counter(a for a in samples["đáp án"] if a)
    print("bỏ phiếu:", dict(votes.most_common()), f"| {len(samples)} mẫu")
    print("đáp án cuối:", json.dumps(q["final"], ensure_ascii=False), "| tỷ lệ đồng thuận:", round(q["conf"], 2))

In [ ]:
if trace and (env.data / SPLIT / "answers.jsonl").exists():
    from datamate import scoring
    gold = {a["id"]: a for a in read(env.data / SPLIT / "answers.jsonl")}
    report, _ = scoring.evaluate({r["id"]: gold[r["id"]] for r in trace}, {r["id"]: r["final"] for r in trace},
                                 {q["id"]: q for q in read(env.data / SPLIT / "questions.jsonl")})
    print({k: report[k] for k in ("N", "correct", "wrong", "abstain", "PA", "precision")})

## 03 · Đánh giá

Điểm của cuộc thi là **PA (penalized accuracy)**: câu đúng +1, câu sai −1, câu bỏ trống 0, rồi chia cho số câu. Vì câu sai bị trừ điểm,
bỏ trống những câu không chắc thường có lợi hơn là đoán.

Phần này gồm:
1. Chấm một lượt chạy trên **train / holdout** (có đáp án): điểm tổng, theo shop, theo dạng câu, nhóm lỗi và ngưỡng bỏ phiếu.
2. So **hai adapter trên cùng câu hỏi** (adapter BF16 mới với `sql_rft_v3`): chênh lệch PA và khoảng tin cậy bootstrap theo shop.
3. So các **bản nộp public** với nhau, không cần nhãn.

In [ ]:
RUN = "runs/nb_train_qwen35_sft_a6000"    # lượt chạy có nhãn (thư mục có trace.jsonl hoặc shard*/trace.jsonl), ví dụ adapter mới trên holdout
BASELINE = "runs/nb_train_sql_rft_v3"     # lượt chạy để so trên cùng câu hỏi (None = bỏ qua phần 2)
SUBS = {"v6 (0.63)": "runs/sub_v6", "v8 (0.639)": "runs/sub_v8", "v9 (0.683)": "runs/sub_v9"}   # bản nộp public cần so

In [ ]:
from datamate import scoring, voting
from datamate.hints import DEBT_WORDS
from datamate.ingest import norm


def load_trace(run):
    run = env.path(run)
    files = [run / "trace.jsonl"] if (run / "trace.jsonl").exists() else sorted(run.glob("shard*/trace.jsonl"))
    return {r["id"]: r for p in files for r in read(p)}


gold = {a["id"]: a for a in read(env.split("train") / "answers.jsonl")}
questions = {q["id"]: q for q in read(env.data / "train/questions.jsonl")}
trace = load_trace(RUN)
print(len(trace), "câu trong", env.path(RUN), "" if trace else "=> chạy 02_inference với SPLIT = 'train' trước")

### 1. Điểm của lượt chạy

- **PA:** điểm của cuộc thi. **precision:** tỷ lệ đúng trong các câu có trả lời. **coverage:** tỷ lệ câu có trả lời.
- **Khoảng tin cậy:** lấy mẫu lại theo shop (bootstrap), cho biết điểm dao động bao nhiêu nếu đổi sang tập shop khác.

In [ ]:
if trace:
    report, rows = scoring.evaluate({i: gold[i] for i in trace}, {i: r["final"] for i, r in trace.items()}, questions)
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in report.items() if k in ("N", "correct", "wrong", "abstain", "PA", "precision", "coverage")})
    print("PA 95% CI (shop bootstrap):", [round(x, 3) for x in report["PA_cluster_bootstrap_95"]])
    display(pd.DataFrame(report["breakdowns"]["gold_type"]).T[["N", "correct", "wrong", "abstain", "PA"]].sort_values("N", ascending=False))

In [ ]:
if trace:
    by_shop = pd.DataFrame(report["breakdowns"]["business_id"]).T[["N", "correct", "wrong", "abstain", "PA"]].sort_values("PA")
    by_shop.plot.barh(y="PA", figsize=(6, max(2, 0.25 * len(by_shop))), legend=False, title="PA by shop")
    display(by_shop.head(8))

#### Nhóm lỗi

Mỗi câu chưa đúng thuộc một nhóm: không mẫu nào đúng (cần model / dữ liệu SFT tốt hơn), có mẫu đúng nhưng nhóm sai thắng phiếu, hoặc nhóm đúng
đứng đầu nhưng dưới ngưỡng nên bỏ trống (cần bỏ phiếu / ngưỡng tốt hơn). Fine-tuning tác động chủ yếu lên nhóm đầu.

In [ ]:
def kind(r):
    g = gold[r["id"]]
    s = scoring.score(r["final"], g)
    if s == 1: return "right"
    if not any(p and scoring.score(p, g) == 1 for p in (x["pred"] for x in r["samples"])): return "no right sample"
    if s == -1: return "wrong: a wrong group won"
    top_right = bool(r.get("vote")) and scoring.score(r["vote"], g) == 1
    return "abstain: right group on top, below threshold" if top_right else "abstain: right sample, another group on top"


pd.Series(Counter(kind(r) for r in trace.values())).sort_values(ascending=False).to_frame("questions")

#### Ngưỡng bỏ phiếu và oracle@K

- **Ngưỡng bỏ phiếu:** tỷ lệ mẫu đồng thuận tối thiểu để trả lời. Ngưỡng cao thì ít sai hơn nhưng bỏ trống nhiều hơn. Bảng bỏ phiếu lại trên các
  mẫu đã lưu với từng ngưỡng.
- **oracle@K:** tỷ lệ câu có ít nhất một mẫu đúng trong K mẫu, tức mức điểm tối đa nếu luôn chọn được mẫu đúng.

In [ ]:
def replay(r, tau):
    texts, preds = [s["text"] for s in r["samples"]], [s["pred"] for s in r["samples"]]
    pool = voting.debt_pool(texts, 8, bool(DEBT_WORDS.search(norm(r["question"]))))
    ans, conf = voting.vote([preds[j] for j in pool], len(pool))
    return voting.final_answer(ans, conf, tau, r["final"].get("reason") == "period outside the shop's data dates", True)


if trace:
    sweep = pd.Series({t: sum(scoring.score(replay(r, t), gold[i]) for i, r in trace.items()) / len(trace) for t in (0.25, 0.375, 0.5, 0.625, 0.75)}, name="PA")
    oracle = sum(any(p and scoring.score(p, gold[i]) == 1 for p in (s["pred"] for s in r["samples"])) for i, r in trace.items()) / len(trace)
    print(f"oracle (any sample right): {oracle:.3f}")
    display(sweep.to_frame())

#### Theo dạng câu hỏi

In [ ]:
from datamate import hints as H

shape = lambda q: ("debt" if DEBT_WORDS.search(norm(q)) else "profit" if H.PROFIT.search(norm(q)) else "trips" if H.TRIPS.search(norm(q))
                   else "lots" if H.LOT.search(norm(q)) else "other")
df = pd.DataFrame([{"shape": shape(r["question"]), "score": scoring.score(r["final"], gold[i])} for i, r in trace.items()])
df.groupby("shape")["score"].agg(n="size", PA="mean").sort_values("n", ascending=False).round(3) if len(df) else None

### 2. So hai adapter trên cùng câu hỏi

Chỉ tính các câu có ở cả hai lượt chạy. **Chênh lệch PA** = PA(`RUN`) − PA(`BASELINE`); khoảng tin cậy 95% lấy mẫu lại theo shop (các câu của
một shop đi cùng nhau), nên phản ánh dao động khi đổi sang shop khác. Bảng chuyển đổi cho biết bao nhiêu câu đổi từ đúng sang sai và ngược lại.

Lưu ý khi đọc trên holdout: `sql_rft_v3` đã học 390 câu holdout nên điểm của nó bị thổi phồng; adapter train trên dữ liệu không có shop holdout
(`04_training_data`, `EXCLUDE_HOLDOUT = True`) mà vẫn ngang hoặc hơn v3 là bằng chứng mạnh.

In [ ]:
import random

base = load_trace(BASELINE) if BASELINE else {}
common = sorted(set(trace) & set(base))
if common:
    new = {i: scoring.score(trace[i]["final"], gold[i]) for i in common}
    old = {i: scoring.score(base[i]["final"], gold[i]) for i in common}
    shop = {i: questions[i]["business_id"] for i in common}
    shops = sorted(set(shop.values()))
    per_shop = pd.DataFrame([{"shop": s, "N": sum(shop[i] == s for i in common), "PA baseline": sum(old[i] for i in common if shop[i] == s),
                              "PA run": sum(new[i] for i in common if shop[i] == s)} for s in shops]).set_index("shop")
    per_shop[["PA baseline", "PA run"]] = per_shop[["PA baseline", "PA run"]].div(per_shop["N"], axis=0)
    per_shop["delta"] = per_shop["PA run"] - per_shop["PA baseline"]
    delta = (sum(new.values()) - sum(old.values())) / len(common)
    rng, boot = random.Random(0), []
    by = {s: [i for i in common if shop[i] == s] for s in shops}
    for _ in range(2000):
        pick = [i for s in rng.choices(shops, k=len(shops)) for i in by[s]]
        boot.append((sum(new[i] for i in pick) - sum(old[i] for i in pick)) / len(pick))
    boot.sort()
    print(f"{len(common)} câu chung, {len(shops)} shop | PA baseline {sum(old.values()) / len(common):.4f} -> run {sum(new.values()) / len(common):.4f} "
          f"| delta {delta:+.4f}, 95% CI [{boot[50]:+.4f}, {boot[1949]:+.4f}]")
    name = {1: "right", 0: "abstain", -1: "wrong"}
    display(pd.crosstab(pd.Series([name[old[i]] for i in common], name="baseline"), pd.Series([name[new[i]] for i in common], name="run")))
    display(per_shop.round(3).sort_values("delta"))
else:
    print("không có câu chung giữa RUN và BASELINE (hoặc BASELINE = None)")

### 3. So các bản nộp public (không nhãn)

Số câu trả lời, mức trùng đáp án giữa các file, và chỗ khác nhau theo shop giữa hai bản cuối.

In [ ]:
def key(a):
    t, x = a.get("type"), a.get("answer")
    if t == "abstain" or (x is None and t != "unanswerable"): return "ABSTAIN"
    return "UNANSWERABLE" if t == "unanswerable" else json.dumps([t, round(x, 2) if isinstance(x, float) else x], ensure_ascii=False, sort_keys=True)


pq = {q["id"]: q for q in read(env.split("public_test") / "questions.jsonl")}
subs = {name: {r["id"]: key(r) for r in read(env.path(p) / "submission.jsonl")} for name, p in SUBS.items() if (env.path(p) / "submission.jsonl").exists()}
print("bản nộp tìm thấy:", list(subs) or "không có")
pd.DataFrame({n: {"answered": sum(v != "ABSTAIN" for v in s.values()), "unanswerable": sum(v == "UNANSWERABLE" for v in s.values())} for n, s in subs.items()}).T

In [ ]:
names = list(subs)
pd.DataFrame([[sum(subs[a].get(i) == subs[b].get(i) for i in pq) / len(pq) for b in names] for a in names], index=names, columns=names).style.format("{:.1%}")

In [ ]:
if len(names) >= 2:
    a, b = names[-2], names[-1]
    display(pd.DataFrame([{"shop": q["business_id"], "changed": subs[a].get(i) != subs[b].get(i), "newly answered": subs[a].get(i) == "ABSTAIN" != subs[b].get(i),
                           "newly abstained": subs[b].get(i) == "ABSTAIN" != subs[a].get(i)} for i, q in pq.items()]).groupby("shop").sum())

## 04 · Dữ liệu SFT cho LoRA

Dựng dữ liệu **supervised fine-tuning** từ các lượt chạy có nhãn trên **train** (rejection sampling), rồi chia train / dev **theo shop**.

> Không bắt buộc: bộ SFT có sẵn trong `data/data/` (6.363 / 1.064 dòng, đã chia theo shop) là dữ liệu mặc định của `05_train_lora`.
> Notebook này là cách thứ hai, dựng dữ liệu từ lượt chạy của chính pipeline trên dữ liệu BTC (cần `data/train` và GPU cho bước 1).

| Bước | Script | Cần | Kết quả |
|---|---|---|---|
| 1. Lượt chạy có nhãn | `02_inference` với `SPLIT = "train"`, `REUSE_LAYERS = False`, `SAVE_PROMPTS = True`, `IDS = "splits/rft_train24.txt"` | GPU bf16 ≥ 24 GB | 8 mẫu thường + tối đa 4 mẫu thinking mỗi câu, kèm prompt đã gửi |
| 2. Chọn lời giải đúng | `training/build_sft.py` | CPU | `all.jsonl`: mỗi câu giữ lời giải đúng trên route tốt nhất (macro công nợ → view chuẩn → bảng thô); câu khó (0-5/8 đúng) tới 2 dòng, câu dễ góp `EASY` dòng, mẫu thinking đúng thành dòng thinking |
| 3. Ghi chú trước SQL | `training/add_notes.py` | GPU (vLLM) | `all_notes.jsonl`: dòng thường có vài dòng `Notes:` trước SQL, đúng định dạng `--notes` lúc suy luận |
| 4. Chia theo shop | `training/split_sft.py` | CPU | `train.jsonl`, `dev.jsonl`, `split_manifest.json` |
| 5. Kiểm tra chất lượng | notebook này | CPU | phân bố route / thinking / shop, độ dài, trùng lặp, định dạng trả lời |

**Vì sao chia theo shop:** các dòng của một shop dùng chung README, schema và thường cả mẫu SQL. Chia ngẫu nhiên theo dòng đặt các bản gần
giống nhau ở cả hai phía, dev loss khi đó không còn đo khả năng tổng quát sang shop mới (điều public / private đòi hỏi).

**Vì sao bỏ shop holdout:** 6 shop của `splits/holdout600.txt` không vào train cũng không vào dev, nên chạy pipeline trên holdout là phép đo sạch
cho adapter mới. LoRA v3 đã học 390 câu holdout nên điểm holdout của nó chỉ là smoke test (`docs/leaderboard.md`).

Chỉ dùng dữ liệu train: `split_sft.py` từ chối mọi dòng có id không phải câu train. Câu public / private không bao giờ vào dữ liệu train.

In [ ]:
install("llm")

In [ ]:
RUNS = ["runs/nb_train_sql_rft_v3"]   # lượt chạy có nhãn trên train với SAVE_PROMPTS (thư mục có trace.jsonl hoặc shard*/trace.jsonl)
TEACHER_RUNS = []                     # tùy chọn: lượt chạy của model lớn hơn trên các câu train mà 9B chưa giải được
OUT_DIR = "runs/sft/sql_rft_nb"      # all.jsonl, all_notes.jsonl, train.jsonl, dev.jsonl, split_manifest.json
EASY = 800                            # số dòng lấy từ câu dễ (6-8 trên 8 mẫu đúng)
MAX = 4000                            # trần số dòng (dòng công nợ và teacher luôn được giữ)
ADD_NOTES = True                      # cần GPU bf16 >= 24 GB (vLLM, trọng số gốc)
DEV_SHOPS = 2                         # số shop làm dev, hoặc danh sách (ví dụ ["biz_104", "biz_119"]); 0 = fit cuối trên mọi shop
EXCLUDE_HOLDOUT = True                # bỏ dòng của 6 shop holdout (splits/holdout_shops.txt) khỏi cả train và dev
SEED = 0                              # chọn shop dev
BUILD_BANK = False                    # ghi thêm ngân hàng few-shot từ RUNS

In [ ]:
def trace_dir(r):
    """build_sft.py đọc RUN/trace.jsonl: lượt chạy nhiều GPU được gộp shard trước."""
    r = env.path(r)
    if not (r / "trace.jsonl").exists() and list(r.glob("shard*/trace.jsonl")):
        with (r / "trace.jsonl").open("w", encoding="utf-8") as f:
            for p in sorted(r.glob("shard*/trace.jsonl")):
                f.write(p.read_text(encoding="utf-8"))
    if not (r / "trace.jsonl").exists():
        raise FileNotFoundError(f"{r}: không có trace.jsonl. Chạy 02_inference với SPLIT = 'train', SAVE_PROMPTS = True trước")
    return r


gold = {a["id"]: a for a in read(env.split("train") / "answers.jsonl")}
q_shop = {q["id"]: q["business_id"] for q in read(env.data / "train/questions.jsonl")}
holdout_shops = set((env.root / "splits/holdout_shops.txt").read_text().split())
runs, teachers = [trace_dir(r) for r in RUNS], [trace_dir(r) for r in TEACHER_RUNS]
summary = []
for r in runs + teachers:
    t = read(r / "trace.jsonl")
    summary.append({"run": r.name, "questions": len(t), "train ids": sum(x["id"] in gold for x in t),
                    "saved prompts": sum(bool(x.get("prompt_user")) for x in t), "holdout-shop questions": sum(q_shop.get(x["id"]) in holdout_shops for x in t),
                    "samples / question": round(sum(len(x["samples"]) for x in t) / max(len(t), 1), 1)})
summary = pd.DataFrame(summary)
display(summary)
assert (summary["train ids"] == summary["questions"]).all(), "chỉ dùng lượt chạy trên train (có đáp án)"
assert (summary["saved prompts"] > 0).all(), "lượt chạy thiếu prompt: chạy lại 02_inference với SAVE_PROMPTS = True"

In [ ]:
out_dir = env.path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
all_rows = out_dir / "all.jsonl"
env.run([env.python, "training/build_sft.py", *runs, *(["--teacher", *teachers] if teachers else []), "--fewshot", "assets/fewshot_bank.jsonl",
         "--out", all_rows, "--max", MAX, "--easy", EASY], log=out_dir / "build_sft.log", show="available|rows by|Traceback|Error")

#### Ghi chú trước SQL

Lúc suy luận, mẫu thường viết vài dòng `Notes:` (khung thời gian, bộ lọc, đơn vị) trước câu SQL (`--notes`). Dòng SFT cần đúng định dạng đó,
nếu không adapter học một định dạng khác với lúc dùng. `add_notes.py` sinh ghi chú bằng model gốc (greedy) và giữ nguyên dòng khi câu trả lời
sai định dạng.

In [ ]:
sft_all = all_rows
if ADD_NOTES and env.can_run_llm:
    flags, _ = env.inference_flags()                      # Ampere: bf16, Ada / Hopper / Blackwell: fp8 như lúc suy luận
    sft_all = out_dir / "all_notes.jsonl"
    env.run([env.python, "training/add_notes.py", all_rows, sft_all, "--model", env.base_model(), "--rulebook", env.data / "train/rulebook.md",
             "--quant", "fp8" if "--quant" in flags else "none"], log=out_dir / "add_notes.log", show="notes on|Traceback|Error")
elif ADD_NOTES:
    try:                                                  # không có GPU: tệp ghi chú đã chạy sẵn trên GPU 24 GB (runs/saved), nếu có
        sft_all = env.use_saved("all_notes.jsonl", out_dir / "all_notes.jsonl")
    except RuntimeError as e:
        print("bỏ qua bước ghi chú:", e, "\n=> dòng thường không có 'Notes:', khác định dạng --notes lúc suy luận")
print("dữ liệu SFT trước khi chia:", sft_all, "|", sum(1 for _ in open(sft_all, encoding="utf-8")), "dòng")

#### Chia train / dev theo shop

`DEV_SHOPS` shop (chọn ngẫu nhiên với `SEED` trong nhóm shop có số dòng ở khoảng giữa) làm dev: dev loss trong lúc train và chọn checkpoint
tốt nhất ở `05_train_lora`. Sau khi đã chọn số bước, có thể train lại trên mọi shop: `DEV_SHOPS = 0` ở đây, `DEV = None` và
`MAX_STEPS` = số bước của checkpoint tốt nhất ở `05_train_lora`.

In [ ]:
dev_arg = ",".join(DEV_SHOPS) if isinstance(DEV_SHOPS, (list, tuple)) else str(DEV_SHOPS)
env.run([env.python, "training/split_sft.py", sft_all, "--out-dir", out_dir, "--dev-shops", dev_arg, "--seed", SEED,
         "--holdout-shops", "splits/holdout_shops.txt" if EXCLUDE_HOLDOUT else ""], show="rows in|split ->|Traceback|Error")
manifest = json.loads((out_dir / "split_manifest.json").read_text(encoding="utf-8"))
pd.DataFrame({s: {"rows": manifest[s]["rows"], "questions": manifest[s]["questions"], "thinking rows": manifest[s]["thinking_rows"],
                  "shops": len(manifest[s]["shops"])} for s in ("train", "dev")}).T

#### Kiểm tra chất lượng dữ liệu

- **Route:** `macro` (macro công nợ), `view` (view chuẩn `v_sales` / `v_orders`), `raw` (bảng thô). Lời giải trên view / macro ngắn và ít sai hơn.
- **Thinking:** dòng thinking dạy chế độ suy nghĩ (dùng cho câu khó). Tỷ lệ quá cao làm adapter thiên về câu trả lời dài.
- **Độ dài:** prompt chiếm phần lớn mỗi dòng (rulebook + schema), loss chỉ tính trên phần trả lời.
- **Định dạng:** câu trả lời phải có khối ```` ```sql ````; dòng thường nên bắt đầu bằng `Notes:`. `05_train_lora` kiểm tra lại bằng tokenizer thật.

In [ ]:
import hashlib

data = {s: read(out_dir / f"{s}.jsonl") for s in ("train", "dev")}
df = pd.DataFrame([{"split": s, "id": r["id"], "shop": q_shop[r["id"]], "route": r.get("route"), "thinking": bool(r.get("thinking")),
                    "prompt_chars": len(r["system"]) + len(r["user"]), "reply_chars": len(r["reply"]),
                    "notes": r["reply"].startswith("Notes:"), "sql_block": "```sql" in r["reply"].split("</think>")[-1],
                    "key": hashlib.sha1("\0".join((r["system"], r["user"], r["reply"])).encode()).hexdigest()}
                   for s, rows in data.items() for r in rows])
display(df.groupby(["split", "route", "thinking"]).size().unstack(fill_value=0))
display(df.groupby("split")[["prompt_chars", "reply_chars"]].describe().round(0))
plain = df[~df.thinking]
print(f"dòng thường có Notes: {plain.notes.mean():.1%} | có khối ```sql: {df.sql_block.mean():.1%} | "
      f"dòng trùng lặp: {df.duplicated('key').sum()} | câu hỏi có > 1 dòng: {(df.groupby('id').size() > 1).sum()}")
assert not set(df[df.split == "train"].shop) & set(df[df.split == "dev"].shop), "train và dev chung shop"
assert not set(df.shop) & holdout_shops or not EXCLUDE_HOLDOUT, "còn dòng của shop holdout"
df.groupby(["shop", "split"]).size().unstack(fill_value=0).plot.barh(stacked=True, figsize=(7, 6), title="rows per shop");

In [ ]:
r = data["train"][0]
print(r["user"][-600:], "\n---- reply ----\n", r["reply"][:800])

In [ ]:
if BUILD_BANK:
    env.run([env.python, "training/build_fewshot_bank.py", *runs, "--out", out_dir / "fewshot_bank.jsonl"], show="Traceback|Error|bank|rows")
print("05_train_lora: TRAIN =", repr(str(out_dir / "train.jsonl")), "| DEV =", repr(str(out_dir / "dev.jsonl")) if manifest["dev"]["rows"] else None)

## 05 · Fine-tune Qwen3.5-9B: SFT LoRA BF16

Train một LoRA adapter cho Qwen3.5-9B bằng **supervised fine-tuning**. Dữ liệu mặc định: bộ SFT trong `data/` (`data/data/train.jsonl`, `dev.jsonl`,
`sft_manifest.json`: 6.363 / 1.064 dòng, đã chia train / dev theo shop), hoặc lời giải đúng của tập train từ `04_training_data`.
Cùng công thức chạy một lệnh, không cần notebook: `bash training/train_a6000.sh`.
Script: `training/train_lora_bf16.py`; kiểm thử trên một Qwen3.5 tí hon: `tests/test_train_lora_bf16.py`.

| Thành phần | Thiết lập | Lý do |
|---|---|---|
| Mô hình gốc | Qwen3.5-9B nạp **bf16** đầy đủ, đóng băng | adapter khớp đúng trọng số vLLM dùng lúc suy luận. QLoRA 4-bit (v1-v3, `training/train_lora.py`) học cả phần bù sai số lượng tử NF4 vốn không có lúc suy luận |
| LoRA | r 16, alpha 32, dropout 0 trên 248 module: q/k/v/o (8 lớp full attention), in_proj_qkv/z/b/a + out_proj (24 lớp Gated DeltaNet), gate/up/down (MLP cả 32 lớp) | cùng module và tên tensor với `adapters/sql_rft_v3`: vLLM (`--lora`, max rank 64) dùng ngay, tiếp tục được từ v3 |
| Prompt | chat template của chính model, `enable_thinking` theo từng dòng; prompt và trả lời tokenize riêng | đúng chuỗi token vLLM nhận lúc suy luận |
| Loss | cross-entropy trung bình trên phần trả lời (+ `<\|im_end\|>`) của mỗi dòng, prompt mask -100 | prompt (rulebook + schema) chiếm ~96% token |
| LM head | chỉ trên các vị trí dự đoán token trả lời, tính theo khối (`--ce-chunk`) | cùng loss và gradient, không tạo tensor logit [độ dài trả lời × 248k] float32 |
| Tối ưu | AdamW fused, lr 1e-4 (5e-5 khi tiếp tục adapter), cosine, warmup 3%, clip 1.0, 16 dòng mỗi bước | |
| Chọn checkpoint | dev loss trên các shop không train, adapter cuối = checkpoint dev loss thấp nhất, dừng sớm | dev loss trên một tập con cố định `DEV_MAX_ROWS` dòng: mỗi dòng ~13k token, đánh giá cả dev tốn ~30% thời gian train |
| Bộ nhớ | gradient checkpointing, micro-batch 1, dòng dài hơn `MAX_LEN` bị loại (không cắt) | dòng bị cắt mất phần trả lời ở cuối |

**Phần cứng:** GPU bf16 (Ampere trở lên) **từ 40 GB** mỗi tiến trình: A100 40/80 GB, L40S / A6000 / RTX 6000 Ada 48 GB, H100. Trọng số bf16 chiếm
~18 GiB, phần còn lại cho activation của các dòng 9k-32k token. GPU 24 GB không đủ cho BF16 với dữ liệu này. Nhiều GPU: data parallel
(`torchrun`), mỗi GPU một bản bf16, số dòng mỗi bước = `GRAD_ACCUM` × số GPU.

**Các bước:** (1) kiểm tra dữ liệu bằng tokenizer, chọn `MAX_LEN` → (2) kiểm tra rò rỉ train / dev / holdout → (3) kiểm tra kernel fla →
(4) preflight bộ nhớ trên các dòng dài nhất → (5) train, checkpoint định kỳ, chạy lại để tiếp tục → (6) đường loss, kiểm tra adapter → (7) dùng adapter.

In [ ]:
install("train")  # Kaggle / Colab: chạy phần này trong một phiên riêng (stack train BF16 khác vLLM)

In [ ]:
TRAIN = "data/data/train.jsonl"             # bộ SFT trong data/ (đã chia theo shop, có sft_manifest.json); hoặc runs/sft/<x>/train.jsonl của 04_training_data
DEV = "data/data/dev.jsonl"                 # shop không train: dev loss, chọn checkpoint, dừng sớm. None = fit cuối trên mọi shop
OUT = "runs/lora/qwen35_sft_a6000"          # checkpoint-*/, adapter/, data_report.json, run_manifest.json, nhật ký
INIT_ADAPTER = None                         # None = adapter mới. "sql_rft_v3" = tiếp tục bản đã nộp (v3 đã học câu holdout: holdout không còn sạch)
MAX_LEN = None                              # token mỗi dòng; None = mức nhỏ nhất (16384 .. 40960) giữ được >= KEEP_SHARE số dòng train (data/: 16384, giữ 100%)
KEEP_SHARE = 0.99
R, ALPHA, DROPOUT = 16, 32, 0.0             # r 16 / alpha 32: tiếp tục được từ sql_rft_v3; vLLM nhận tới rank 64
LR = None                                   # None = 1e-4 (adapter mới) hoặc 5e-5 (tiếp tục INIT_ADAPTER)
EPOCHS = 2                                  # checkpoint dev loss tốt nhất được giữ, dừng sớm khi dev loss thôi giảm
MICRO_BATCH, GRAD_ACCUM = 1, 16             # 16 dòng mỗi bước tối ưu trên mỗi GPU (data/: ~398 bước mỗi epoch)
WARMUP_RATIO, SCHEDULER, WEIGHT_DECAY, MAX_GRAD_NORM = 0.03, "cosine", 0.0, 1.0
EVAL_STEPS, SAVE_STEPS = 50, 50             # SAVE_STEPS phải là bội của EVAL_STEPS
EARLY_STOP = 3                              # dừng sau 3 lần đánh giá liên tiếp không giảm dev loss (0 = tắt)
DEV_MAX_ROWS = 320                          # dev loss trên 320 dòng dev cố định (0 = cả dev; data/ có 1.064 dòng ~13k token)
CE_CHUNK = 1024                             # LM head theo khối 1024 vị trí trả lời (0 = head thường, tốn bộ nhớ hơn với câu trả lời dài)
ATTN = "sdpa"                               # 8 lớp full attention: sdpa, hoặc flash_attention_2 khi đã cài flash-attn
NUM_GPUS = None                             # None = mọi GPU thấy được
LIMIT_ROWS, MAX_STEPS = None, -1            # chạy thử nhanh: ví dụ LIMIT_ROWS = 64, MAX_STEPS = 10
PREFLIGHT = True                            # 2 bước trên các dòng dài nhất trước khi train thật: đo đỉnh bộ nhớ ở MAX_LEN
RESUME = True                               # tiếp tục từ checkpoint mới nhất trong OUT (cùng dữ liệu và thiết lập)
SEED = 3407
REPORT_TO = "tensorboard"                   # none | tensorboard | wandb

In [ ]:
train_file, dev_file, out = env.path(TRAIN), env.path(DEV) if DEV else None, env.path(OUT)
for f in [train_file] + ([dev_file] if dev_file else []):
    assert f.exists(), f"{f} chưa có: đặt bộ SFT vào data/data/ hoặc chạy 04_training_data trước"
init = env.adapter(INIT_ADAPTER) if INIT_ADAPTER else None
lr = LR or (5e-5 if init else 1e-4)
venv_train = env.root / ".venv-train/bin/python"
py = str(venv_train) if venv_train.exists() else env.python      # stack train (torch, transformers, peft, fla) tách khỏi vLLM
n_gpu = NUM_GPUS or max(1, len(env.gpus))
train_env = {"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", "TOKENIZERS_PARALLELISM": "false"}
print(f"python train: {py}\nGPU: {[f'{g.name} {g.mem_gb} GB' for g in env.gpus] or 'không có'} | BF16 LoRA chạy được ở đây: {env.can_train_bf16} "
      f"(cần bf16 và >= 40 GB mỗi GPU)\nlr {lr} | init {init or 'adapter mới'} | {n_gpu} GPU x {MICRO_BATCH} x {GRAD_ACCUM} = "
      f"{n_gpu * MICRO_BATCH * GRAD_ACCUM} dòng mỗi bước")

#### Kiểm tra rò rỉ

- Mọi dòng phải là câu **train** (có đáp án), không bao giờ là câu public / private. Bộ SFT trong `data/` là dữ liệu tổng hợp
  (`metadata.dataset_role = synthetic_train`, shop `syn_shop_*`): không chứa câu nào của BTC, mã câu hỏi và shop nằm ngay trong dòng.
- Train và dev **không chung câu hỏi, không chung shop** (dev loss đo khả năng sang shop mới).
- Không có câu của **holdout** (`splits/holdout600.txt`), để chạy pipeline trên holdout với adapter mới là phép đo sạch. Tiếp tục từ
  `sql_rft_v3` cũng làm holdout không còn sạch (v3 đã học 390 câu holdout).

In [ ]:
holdout_ids = set((env.root / "splits/holdout600.txt").read_text().split())
rows = {"train": read(train_file), "dev": read(dev_file) if dev_file else []}
if any("prompt_messages" in r for rs in rows.values() for r in rs):
    # bộ SFT của data/ (dữ liệu tổng hợp): mã câu hỏi và shop ở ngay trong dòng
    roles = Counter((r.get("metadata") or {}).get("dataset_role") for rs in rows.values() for r in rs)
    assert set(roles) == {"synthetic_train"}, f"có dòng không thuộc dữ liệu tổng hợp để train: {dict(roles)}"
    ids = {s: {r["metadata"]["question_id"] for r in rs} for s, rs in rows.items()}
    shops = {s: {r["business_id"] for r in rs} for s, rs in rows.items()}
else:
    # dữ liệu của 04_training_data: mọi dòng phải là câu train của BTC
    q_shop = {q["id"]: q["business_id"] for q in read(env.split("train") / "questions.jsonl")}
    ids = {s: {str(r["id"]) for r in rs} for s, rs in rows.items()}
    shops = {s: {q_shop.get(i) for i in v} for s, v in ids.items()}
    unknown = (ids["train"] | ids["dev"]) - set(q_shop)
    assert not unknown, f"{len(unknown)} dòng không phải câu train, ví dụ {sorted(unknown)[:3]}"
assert not ids["train"] & ids["dev"], f"train và dev chung câu hỏi: {sorted(ids['train'] & ids['dev'])[:3]}"
assert not shops["train"] & shops["dev"], f"train và dev chung shop: {sorted(shops['train'] & shops['dev'])}"
leak = (ids["train"] | ids["dev"]) & holdout_ids
holdout_clean = not leak and init is None
display(pd.DataFrame({s: {"rows": len(rows[s]), "questions": len(ids[s]), "shops": len(shops[s]),
                          "thinking rows": sum(bool(r.get("thinking", r.get("thinking_mode"))) for r in rows[s]),
                          "holdout questions": len(ids[s] & holdout_ids)} for s in rows}).T)
print("holdout sạch cho adapter này:" , holdout_clean, "" if holdout_clean else
      f"({len(leak)} câu holdout trong dữ liệu)" if leak else "(tiếp tục từ adapter đã học câu holdout)")
del rows   # ~350 MB JSON: trainer tự đọc lại


#### 1. Kiểm tra dữ liệu bằng tokenizer

Chỉ cần tokenizer và chat template (không cần trọng số, không cần GPU): mỗi dòng được render đúng như lúc suy luận, kiểm tra định dạng
(khối ```` ```sql ````, đúng một `</think>` ở dòng thinking, không có token đặc biệt trong câu trả lời), trùng lặp, tokenize khứ hồi, độ dài.
Báo cáo `data_report.json` cho biết tỷ lệ dòng giữ được ở từng `MAX_LEN`; `MAX_LEN = None` chọn mức nhỏ nhất giữ được `KEEP_SHARE` số dòng
(dòng dài hơn bị loại, không bị cắt).

In [ ]:
check_dir = out.with_name(out.name + "_data_check")
try:
    tok_src = env.base_model() if env.can_train_bf16 else env.adapter("sql_rft_v3")   # thư mục adapter có cùng tokenizer + chat template
    env.run([py, "training/train_lora_bf16.py", "--train", train_file, *(["--dev", dev_file] if dev_file else []), "--model", tok_src,
             "--tokenizer", tok_src, "--out", check_dir, "--max-len", 65536, "--validate-only"],
            log=check_dir / "validate.log", show="train:|dev:|share of rows|sft_manifest|validate-only|Traceback|Error")
except RuntimeError as e:
    print("không kiểm tra được dữ liệu ở đây (cần transformers của training/requirements_bf16.txt và tokenizer):", e)
report = json.loads((check_dir / "data_report.json").read_text()) if (check_dir / "data_report.json").exists() else None
if report:
    t = report["train"]
    kept = {int(k): v for k, v in t["share_kept_at_max_len"].items()}
    max_len = MAX_LEN or next((L for L in sorted(kept) if L >= 16384 and kept[L] >= KEEP_SHARE), max(kept))
    display(pd.DataFrame({s: {"rows read": report[s]["rows_read"], "rows kept": report[s]["rows_kept"], "label share": f'{report[s]["label_share"]:.1%}',
                              "tokens p50 / p90 / max": "{p50} / {p90} / {max}".format(**report[s]["row_tokens"]),
                              "reply tokens p50 / max": "{p50} / {max}".format(**report[s]["reply_tokens"]), "rejected": report[s]["rejected"] or "-"}
                          for s in ("train", "dev") if report.get(s)}).T)
    ax = pd.Series(kept).plot.bar(figsize=(6, 2.5), title="share of train rows kept at each MAX_LEN", ylim=(0, 1.05))
    ax.axhline(KEEP_SHARE, ls="--", c="gray")
    print(f"MAX_LEN = {max_len}: giữ {kept.get(max_len, float('nan')):.1%} số dòng train")
else:
    max_len = MAX_LEN or 24576

#### 2. Kernel fla

24 trên 32 lớp của Qwen3.5-9B là Gated DeltaNet. Có `fla-core` thì transformers dùng kernel Triton (nhanh hơn khoảng 90 lần mỗi lớp so với bản
PyTorch tham chiếu). Ô này so kernel với bản tham chiếu (forward và backward) trên dữ liệu ngẫu nhiên ở đúng kích thước của model. Trượt thì train
không có fla (đúng nhưng rất chậm).

In [ ]:
extra_env = {}
if not env.can_train_bf16:
    print("không có GPU bf16 >= 40 GB: bỏ qua kiểm tra kernel")
else:
    try:
        env.run([py, "training/check_fla.py", "--T", 4096], show="fla check|T=|rel L2|Traceback|Error")
    except RuntimeError as e:
        print("fla không dùng được ở đây, train không có fla:", e)
        extra_env = {"PYTHONPATH": str(env.root / "training/no_fla")}

In [ ]:
def train_cmd(out_dir, gpus=n_gpu, dev=dev_file, extra=(), report=REPORT_TO):
    launch = [py] if gpus == 1 else [py, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={gpus}"]
    cmd = launch + ["training/train_lora_bf16.py", "--train", train_file, *(["--dev", dev] if dev else []), "--model", model, "--out", out_dir,
                    "--max-len", max_len, "--r", R, "--alpha", ALPHA, "--dropout", DROPOUT, "--lr", lr, "--epochs", EPOCHS,
                    "--micro-batch", MICRO_BATCH, "--grad-accum", GRAD_ACCUM, "--warmup-ratio", WARMUP_RATIO, "--scheduler", SCHEDULER,
                    "--weight-decay", WEIGHT_DECAY, "--max-grad-norm", MAX_GRAD_NORM, "--eval-steps", EVAL_STEPS, "--save-steps", SAVE_STEPS,
                    "--early-stop", EARLY_STOP, "--dev-max-rows", DEV_MAX_ROWS, "--ce-chunk", CE_CHUNK, "--attn", ATTN, "--seed", SEED, "--report-to", report]
    cmd += (["--init-adapter", init] if init else []) + (["--limit-rows", LIMIT_ROWS] if LIMIT_ROWS else [])
    return cmd + (["--max-steps", MAX_STEPS] if MAX_STEPS > 0 and "--max-steps" not in extra else []) + list(extra)


model = env.base_model() if env.can_train_bf16 else None            # Qwen3.5-9B đúng revision đã nộp (~19 GB, tải một lần)
SHOW = "[train]|[eval]|train:|dev:|dev loss on|sft_manifest|Gated DeltaNet kernel|LoRA r=|rows per optimizer|resuming|continuing|steps in|adapter|Traceback|Error|OutOfMemory|out of memory"

#### 3. Preflight bộ nhớ

Hai bước tối ưu, mỗi bước một dòng, trên **hai dòng dài nhất** (`--longest-first --grad-accum 1`), một GPU, không dev: đỉnh bộ nhớ thật ở `MAX_LEN` trước khi bỏ ra nhiều giờ.
Hết bộ nhớ thì giảm `MAX_LEN` (dòng dài hơn bị loại) hoặc dùng GPU lớn hơn; còn dư nhiều thì có thể tăng `MAX_LEN` (giữ nhiều dòng hơn).
Bỏ qua khi đang tiếp tục một lượt đã có checkpoint.

In [ ]:
resuming = RESUME and any(out.glob("checkpoint-*"))                 # tiếp tục một lượt đã chạy: bộ nhớ đã được kiểm chứng
if env.can_train_bf16 and PREFLIGHT and not resuming:
    pre = out.with_name(out.name + "_preflight")
    shutil.rmtree(pre, ignore_errors=True)
    try:
        env.run(train_cmd(pre, gpus=1, dev=None, extra=["--max-steps", 2, "--longest-first", "--grad-accum", 1, "--limit-rows", 0], report="none"),
                log=pre.with_suffix(".log"), env={**train_env, **extra_env}, show=SHOW, every=120)
        m = json.loads((pre / "run_manifest.json").read_text())
        use = m["peak_gpu_reserved_gib"] / m["gpu_total_gib"]
        print(f"preflight: {m['peak_gpu_gib']} GiB allocated, {m['peak_gpu_reserved_gib']} GiB reserved / {m['gpu_total_gib']} GiB ({use:.0%}), "
              f"{m['seconds'] / 2:.0f} s mỗi dòng dài nhất (bước đầu gồm cả autotune kernel)")
        if use > 0.95:
            print("=> sát giới hạn bộ nhớ: giảm MAX_LEN hoặc CE_CHUNK trước khi train thật")
    except RuntimeError as e:
        print(f"preflight thất bại ({e}). Xem {pre.with_suffix('.log')}: hết bộ nhớ thì giảm MAX_LEN (hiện {max_len}).")
        raise

#### 4. Train

Nhật ký đầy đủ ở `OUT.log`; notebook chỉ hiện dòng loss / dev loss. Checkpoint được lưu mỗi `SAVE_STEPS` bước vào `OUT` (Kaggle:
`/kaggle/working/runs`, Colab: Google Drive, giữ lại sau phiên). Phiên bị ngắt: chạy lại notebook với `RESUME = True`, cùng dữ liệu và
thiết lập (`run_identity.json` kiểm tra điều này). Muốn bắt đầu lại từ đầu: đổi `OUT`.

In [ ]:
if env.can_train_bf16:
    env.run(train_cmd(out) + (["--resume"] if RESUME else []), log=out.with_name(out.name + ".log"), env={**train_env, **extra_env}, show=SHOW, every=300)
else:
    print("không có GPU bf16 >= 40 GB: bỏ qua bước train (A100, L40S, H100...). Các ô sau đọc kết quả nếu OUT đã có từ một máy khác.")

#### 5. Đường loss

Train loss theo bước (mờ: từng bước, đậm: trung bình trượt), dev loss ở mỗi lần đánh giá, learning rate. Dev loss tăng trở lại trong khi train
loss vẫn giảm là dấu hiệu overfit: checkpoint tốt nhất (dev loss thấp nhất) là adapter cuối.

In [ ]:
state_file = out / "trainer_state.json"
if not state_file.exists():                                           # đang train hoặc bị ngắt: checkpoint mới nhất
    ckpts = sorted(out.glob("checkpoint-*/trainer_state.json"), key=lambda p: int(p.parent.name.split("-")[1]))
    state_file = ckpts[-1] if ckpts else None
if state_file:
    state = json.loads(state_file.read_text())
    hist = pd.DataFrame(state["log_history"])
    tr = hist.dropna(subset=["loss"]) if "loss" in hist else pd.DataFrame()
    ev = hist.dropna(subset=["eval_loss"]) if "eval_loss" in hist else pd.DataFrame()
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.5))
    if len(tr):
        a1.plot(tr.step, tr.loss, alpha=0.25, label="train loss")
        a1.plot(tr.step, tr.loss.rolling(max(1, len(tr) // 20), min_periods=1).mean(), label="train loss (smoothed)")
        a2.plot(tr.step, tr.learning_rate, c="gray")
    if len(ev):
        a1.plot(ev.step, ev.eval_loss, "o-", label="dev loss")
    a1.set(xlabel="step", title="loss"); a1.legend(); a2.set(xlabel="step", title="learning rate")
    plt.show()
    print(f"bước {state['global_step']} / {state['max_steps']}; dev loss tốt nhất {state.get('best_metric')} ({state.get('best_model_checkpoint')})")
else:
    print(f"chưa có trainer_state.json trong {out}")

#### 6. Kiểm tra adapter

Trước khi đưa vào pipeline: r / alpha, đủ 248 module (496 tensor), tên tensor trùng với `adapters/sql_rft_v3` (vLLM nạp theo tên), và
`lora_B` khác 0 (adapter thật sự đã học). `run_manifest.json` ghi số bước, thời gian, loss, đỉnh bộ nhớ, phiên bản thư viện.

In [ ]:
import struct

import numpy as np


def st_tensors(path, names=None):
    """Header của .safetensors và (tùy chọn) vài tensor float32 đọc thẳng từ tệp: không cần torch trong kernel này."""
    with open(path, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        head = json.loads(f.read(n))
        head.pop("__metadata__", None)
        arrays = {}
        for k in names or []:
            a, b = head[k]["data_offsets"]
            f.seek(8 + n + a)
            arrays[k] = np.frombuffer(f.read(b - a), dtype=np.float32)
    return head, arrays


adapter = out / "adapter"
if (adapter / "adapter_model.safetensors").exists():
    cfg = json.loads((adapter / "adapter_config.json").read_text())
    head, _ = st_tensors(adapter / "adapter_model.safetensors")
    b_keys = [k for k in head if "lora_B" in k and head[k]["dtype"] == "F32"]
    _, b = st_tensors(adapter / "adapter_model.safetensors", b_keys)
    base_cfg = (Path(model) if model else env.models / "Qwen3.5-9B") / "config.json"
    types = (json.loads(base_cfg.read_text())["text_config"]["layer_types"] if base_cfg.exists()
             else (["linear_attention"] * 3 + ["full_attention"]) * 8)                  # Qwen3.5-9B: 24 Gated DeltaNet + 8 full attention
    want = 2 * sum(7 if t == "full_attention" else 8 for t in types)   # lora_A + lora_B: 7 projections per attention layer, 8 per Gated DeltaNet layer
    ref = env.root / "adapters/sql_rft_v3/adapter_model.safetensors"
    same_names = set(head) == set(st_tensors(ref)[0]) if want == 496 and ref.exists() and ref.stat().st_size > 1_000_000 else None
    checks = {"r / alpha": f"{cfg['r']} / {cfg['lora_alpha']}", "tensors": f"{len(head)} / {want}", "names = sql_rft_v3": same_names,
              "lora_B khác 0": f"{sum(bool(np.abs(v).max() > 0) for v in b.values())} / {len(b)}",
              "MB": round((adapter / "adapter_model.safetensors").stat().st_size / 2**20, 1)}
    display(pd.Series(checks, name="adapter").to_frame())
    assert len(head) == want and same_names in (True, None), "adapter không khớp cách vLLM nạp sql_rft_v3"
    if (out / "run_manifest.json").exists():
        m = json.loads((out / "run_manifest.json").read_text())
        print({k: m.get(k) for k in ("steps", "seconds", "train_loss", "best_dev_loss", "best_checkpoint", "stopped_early", "peak_gpu_gib", "gpu", "versions")})
else:
    print(f"chưa có adapter trong {adapter}")

#### 7. Dùng adapter vừa train

1. **Đo trên holdout (sạch, cần dữ liệu BTC trong `data/train`):** `02_inference` với `SPLIT = "train"`, `IDS = "splits/holdout600.txt"`, `REUSE_LAYERS = False`,
   `LORA = "runs/lora/qwen35_sft_a6000/adapter"` (`OUT` + `/adapter`). Chạy thêm một lượt với `LORA = "sql_rft_v3"` làm mốc.
2. **So sánh:** `03_evaluation` với `RUN` = lượt của adapter mới, `BASELINE` = lượt của v3: PA, chênh lệch theo shop, khoảng tin cậy bootstrap.
   v3 đã học 390 câu holdout nên điểm holdout của v3 bị thổi phồng; adapter mới thắng v3 trên holdout là bằng chứng mạnh.
3. **Fit cuối (tùy chọn):** khi đã chọn được số bước, train lại không giữ dev: với bộ SFT `data/data` đặt `DEV = None`; với dữ liệu của
   `04_training_data` đặt `DEV_SHOPS = 0` ở đó rồi `DEV = None` ở đây;
   `EPOCHS` giữ nguyên và `MAX_STEPS` = số bước của checkpoint tốt nhất, `OUT` mới.
4. **Nộp:** chạy `02_inference` trên `public_test` với `LORA = "runs/lora/qwen35_sft_a6000/adapter"`, so với bản trước bằng `03_evaluation` (phần so
   các bản nộp). Chỉ khi điểm public cao hơn bản đang nộp mới làm bước 8.

#### 8. Đưa adapter BF16 vào bản nộp

Chỉ làm khi adapter mới có điểm public (chạy `FRESH=1`) cao hơn bản đang nộp. Chạy trên checkout local (trên Kaggle / Colab code là bản sao).
- `SUBMIT = False`: `scripts/use_adapter.py --dry-run` kiểm adapter (rank, alpha, `target_modules`, model gốc, tổng ≤ 15B) và liệt kê các chỗ sẽ
  sửa (`docker/Dockerfile`, `.dockerignore`, `docker/model_manifest.json`, `scripts/run_public.sh`, `scripts/run_holdout.sh`). Không ghi gì.
- `SUBMIT = True`: chép dữ liệu và cấu hình của lượt train vào `data/sft_bf16/` (train / dev, `split_manifest.json` hoặc `sft_manifest.json`, `train_args.json`, nhật ký)
  để `docker/train.sh` train lại đúng adapter này trong image (code train của bản nộp phải tái tạo được adapter), rồi đổi adapter nộp sang
  `adapters/NAME`. Sau đó: `python3 scripts/rule_check.py`, `python3 scripts/pack_training_code.py`, build Docker, commit (trọng số và
  `data/sft_bf16/*.jsonl` đi qua Git LFS).

In [ ]:
SUBMIT = False                 # True: chép dữ liệu vào data/sft_bf16/ và đổi adapter nộp. False: chỉ kiểm tra (dry-run)
NAME = "qwen35_sft_a6000"      # thư mục mới trong adapters/

adapter = out / "adapter"
if not (adapter / "adapter_model.safetensors").exists():
    print(f"chưa có adapter trong {adapter}")
else:
    if SUBMIT:
        dst = env.root / "data/sft_bf16"
        dst.mkdir(parents=True, exist_ok=True)
        files = {"train.jsonl": train_file, "dev.jsonl": dev_file, "split_manifest.json": train_file.parent / "split_manifest.json",
                 "sft_manifest.json": train_file.parent / "sft_manifest.json",
                 "train_args.json": out / "train_args.json", "run_manifest.json": out / "run_manifest.json", "trainer_state.json": out / "trainer_state.json",
                 "data_report.json": out / "data_report.json", "train.log": out.with_name(out.name + ".log")}
        for name, src in files.items():
            if src and Path(src).exists():
                shutil.copy2(src, dst / name)
        assert (dst / "train_args.json").exists(), "thiếu train_args.json của lượt train: docker/train.sh cần nó"
        print("data/sft_bf16:", sorted(p.name for p in dst.iterdir()))
    env.run([env.python, "scripts/use_adapter.py", adapter, "--name", NAME] + ([] if SUBMIT else ["--dry-run"]),
            show="->|reference|manifest|dry run|done|Error|error|adapters/")